# Exploration du trafic aérien (DGAC, série PAP)

Source : DGAC, trafic aérien commercial mensuel par paire d'aéroports (data.gouv.fr, Licence Ouverte v2).
Années retenues : 2017 et 2023, comme le reste du projet.

In [1]:
import os
import sys

# le notebook est dans notebooks/ : on se place à la racine du projet,
# là où sont data/ et src/ (une seule fois, même si on relance la cellule)
if os.path.basename(os.getcwd()) == "notebooks":  # os.getcwd() permet de savoir où Python travaille
    os.chdir("..")  #  permet de remonter à la racine du projet
sys.path.append(os.getcwd()) #  permet à python d'aller dans src

from src.collecte import get_dgac

url = "https://static.data.gouv.fr/resources/trafic-aerien-commercial-mensuel-francais-par-paire-daeroports-par-sens-depuis-1990/20260903-120658/asp-pap-1990-2025.zip"
pap = get_dgac(url, [2017, 2023])

print(pap.shape)
pap.head(1)

(3572, 13)


,ANMOIS,PAP,PAP1_LABEL,PAP2_LABEL,PAP_DIST,PAP_PEQ,PAP_PAX,PAP_FRP,PAP_DRT,PAP_MVT,PAP_PEQ_FS,PAP_PAX_FS,PAP_FRP_FS
0,201701,FMEE_FIMP,LA REUNION-ROLAND GARROS,MAURICE INTL,232,80196,77169,302.7,462,462,81134,78056,307.7


## Collecte

Les données viennent de la DGAC, publiées sur data.gouv.fr. La fonction `get_dgac` télécharge
l'archive une seule fois dans `data/raw/`, sans jamais la modifier, puis lit seulement 2017 et 2023.
Le projet peut ainsi être relancé par n'importe qui et donne toujours les mêmes données.

Nous utilisons la série PAP (trafic par paire d'aéroports) : c'est la seule qui donne les passagers
par liaison, ce qui permet de comparer chaque ligne aérienne avec le train.

In [2]:
# 1. types et valeurs manquantes de chaque colonne
resume = pap.dtypes.to_frame("type")
resume["manquants"] = pap.isna().sum()
print(resume)

# 2. liaisons intérieures à la métropole : les deux codes OACI commencent par LF
interieur = pap["PAP"].str.match(r"^LF.._LF..$")
print("lignes intérieures à la métropole :", interieur.sum(), "sur", len(pap))

# 3. une même paire apparaît-elle dans les deux sens ? exemple Orly - Nice
print(pap.loc[pap["PAP"].isin(["LFPO_LFMN", "LFMN_LFPO"]), ["ANMOIS", "PAP", "PAP_PAX"]].head(4))

               type  manquants
ANMOIS          str          0
PAP             str          0
PAP1_LABEL      str          0
PAP2_LABEL      str          0
PAP_DIST      int64          0
PAP_PEQ       int64          0
PAP_PAX       int64          0
PAP_FRP     float64          0
PAP_DRT       int64          0
PAP_MVT       int64          0
PAP_PEQ_FS    int64          0
PAP_PAX_FS    int64          0
PAP_FRP_FS  float64          0
lignes intérieures à la métropole : 476 sur 3572
     ANMOIS        PAP  PAP_PAX
94   201701  LFPO_LFMN   155167
324  201703  LFPO_LFMN   171111
623  201705  LFPO_LFMN   200434
781  201706  LFPO_LFMN   208092


## Ce que dit la notice DGAC

- Une ligne correspon à une liaison et un mois, départs et arrivées confondus (pas de doublon par sens).
- Ne sont publiés que les mois avec au moins 10 000 passagers équivalents, au moins 3 compagnies
  et aucune au-dessus de 85 % du trafic (secret statistique).
  Une liaison absente n'a donc pas zéro passager : elle reste NaN.
- On retient `PAP_PAX`, les passagers locaux de la liaison : ce sont les voyageurs qui vont réellement d'une ville à l'autre, c'est-à-dire ceux qui pourraient prendre le train à la place de l'avion.

voici l'url de la notice: https://static.data.gouv.fr/resources/trafic-aerien-commercial-mensuel-francais-par-paire-daeroports-par-sens-depuis-1990/20260903-120427/notice-asp-serie-pap.pdf

In [3]:
# on garde les liaisons intérieures et on ajoute l'année
dom = pap[interieur].copy()
dom["annee"] = dom["ANMOIS"].str[:4]   # ANMOIS vaut par exemple 201702, les quatre premiers caractères sont donc l'année, ce que l'on cherche à isoler et à compter

# pour chaque liaison et chaque année : nombre de mois publiés et passagers
par_liaison = dom.groupby(["PAP", "annee"]).agg(
    mois_publies=("ANMOIS", "count"),
    passagers=("PAP_PAX", "sum"),
).reset_index()   # .reset_index() remet PAP et annee en colonnes normales, pour avoir un tableau simple.

print("liaisons par année :")
print(par_liaison["annee"].value_counts())
print()
print("nombre de mois publiés (combien de liaisons dans chaque cas) :")
print(par_liaison.groupby("annee")["mois_publies"].value_counts().sort_index())

liaisons par année :
annee
2017    41
2023    39
Name: count, dtype: int64

nombre de mois publiés (combien de liaisons dans chaque cas) :
annee  mois_publies
2017   1                3
       2                4
       3                3
       4                3
       5                4
       6                3
       7                5
       8                2
       9                2
       10               1
       11               1
       12              10
2023   1                9
       2                6
       3                5
       4                2
       5                2
       6                2
       7                1
       8                3
       9                2
       10               1
       11               1
       12               5
Name: count, dtype: int64


In [4]:
# numéro du mois (01 à 12), pour comparer un mois avec le même mois
dom["mois"] = dom["ANMOIS"].str[4:]

# une ligne par liaison et par mois, une colonne par année
cote_a_cote = dom.pivot_table(index=["PAP", "mois"], columns="annee", values="PAP_PAX")

# mois publiés à la fois en 2017 et en 2023
communs = cote_a_cote.dropna()

# nombre de mois communs pour chaque liaison
mois_communs = communs.groupby("PAP").size()
print("liaisons avec au moins un mois commun :", len(mois_communs))
print()
print("nombre de mois communs (combien de liaisons dans chaque cas) :")
print(mois_communs.value_counts().sort_index())

liaisons avec au moins un mois commun : 24

nombre de mois communs (combien de liaisons dans chaque cas) :
1     4
2     3
3     1
4     2
5     2
6     2
7     1
8     1
9     2
10    1
11    1
12    4
Name: count, dtype: int64


## Problèmes liés aux traitements des données et au nettoyage de la base

### 1. Le fichier contient tout le trafic, pas seulement les vols intérieurs
Sur les 3 572 observations, seulement 476 relient des aéroports au sein de la métropole. En effet, la DGAC publie tout le trafic qui touche la France (dont vols internationaux, avec les DOM et aussi en métropole). Il y a donc une nécessité de filtrer les vols intra métropoles des autres. Pour cela, il faut garder uniquement les liaisons dont les deux codes OACI commencent par `LF` (France métropolitaine et Corse).

### 2. A cause du secret statistique, il manque plusieurs mois de données pour certains aéroports
Par exemple, Orly – Nice n'a pas de données en février ni en avril 2017, alors que la liaison dépasse largement 10 000 passagers par mois. la DGAC ne publie un mois que s'il y a au moins 3 compagnies
  et qu'aucune ne dépasse 85 % du trafic (notice de la série PAP). Additionner les mois publiés sous-estime le trafic annuel, et fausse la comparaison 2017 / 2023 si le nombre de mois diffère.
Comme solution, un peu pis à lait, on a décider de ne garder que les mois publiés entre les années pour comparer 2017 et 2023, on ne garde, pour chaque liaison, que les mois publiés
  les deux années (par exemple mars 2017 et mars 2023). Un mois caché par le secret statistique
  est ainsi écarté les deux années, et l'effet des saisons s'annule.
  24 liaisons ont au moins un mois en commun (4 en ont 12, 7 en ont seulement 1 ou 2).
  On garde le nombre de mois communs dans une colonne `mois_communs` pour signaler les
  évolutions fragiles. Les liaisons publiées une seule année servent à décrire cette année-là,
  mais pas à calculer une évolution.

## Ce qui n'est pas un problème
- Aucune valeur manquante, codes déjà en texte.
- Chaque liaison n'apparaît que dans un sens, les deux sens étant additionnés par la DGAC
  (vérifié sur Orly – Nice) : rien à regrouper.

In [5]:
from src.nettoyage import clean_dgac

liaisons = clean_dgac(pap, 2017, 2023)
print(liaisons.shape)
print("liaisons avec au moins un mois commun :", (liaisons["mois_communs"] > 0).sum())
liaisons[liaisons["code_liaison"] == "LFPO_LFMN"].T

(52, 13)
liaisons avec au moins un mois commun : 24


,43
code_liaison,LFPO_LFMN
code_oaci_1,LFPO
code_oaci_2,LFMN
nom_aeroport_1,PARIS-ORLY
nom_aeroport_2,NICE-COTE D'AZUR
dist_km,674
mois_publies_2017,8
passagers_2017,1448041.0
mois_publies_2023,8
passagers_2023,1098774.0


In [6]:
# sauvegarde de la table propre
liaisons.to_parquet("data/clean/liaisons_aeriennes.parquet")
print("manquants par colonne :")
print(liaisons.isna().sum())

manquants par colonne :
code_liaison               0
code_oaci_1                0
code_oaci_2                0
nom_aeroport_1             0
nom_aeroport_2             0
dist_km                    0
mois_publies_2017          0
passagers_2017            11
mois_publies_2023          0
passagers_2023            13
mois_communs               0
passagers_communs_2017    28
passagers_communs_2023    28
dtype: int64


Les valeurs manquantes sont celles attendues. En effet, il s'agit de toutes les liaisons non publiées edans la base de données (11 en 2017, 13 en 2023 et 28 sans mois commun). Pour ne pas fausser les données les données manquantes n'ont pas été remplacées par des 0, elles sont donc écartées de l'analyse, cela reste une grosse limite de notre analyse à cause de cette base lacunaire. 

## Bilan

Fichier propre : `data/clean/liaisons_aeriennes.parquet`

- 52 liaisons intérieures à la métropole publiées en 2017 ou en 2023 (41 en 2017, 39 en 2023).
- 24 ont au moins un mois publié les deux années : ce sont elles qui servent à mesurer l'évolution.
- Les passagers d'une année sont un minimum, à cause des mois cachés par le secret statistique.
  Une liaison non publiée reste NaN, jamais 0.

**Exemple : Orly – Nice.** La liaison est publiée 8 mois en 2017 et 8 mois en 2023, dont 5 mois
en commun. Sur ces 5 mois, le trafic passe de 916 663 à 688 391 passagers, soit environ -25 %.
Comparer les totaux annuels (1 448 041 contre 1 098 774) n'aurait pas été juste, car les mois
publiés ne sont pas tous les mêmes d'une année à l'autre.